# 03 — Validate Preprocessing Across Subjects

## Objective

Test the reusable preprocessing pipeline on multiple EEGMMIDB subjects before beginning machine-learning analysis.

The goal is to verify that the same preprocessing procedure produces consistent trial dimensions, valid labels, and usable metadata across subjects, while identifying subjects that require further inspection.

## Imports and project paths

In [1]:
from pathlib import Path
import sys
import mne
import numpy as np
import pandas as pd

mne.set_log_level("WARNING")

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurosignallab.preprocessing import preprocess_subject

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "eegbci"

## Validate for 5 subjects

In [2]:
validation_rows = []

for subject in range(1, 6):

    X, y, _ = preprocess_subject(
        subject=subject,
        data_dir=DATA_DIR
    )

    validation_rows.append({
        "subject": subject,
        "trials": X.shape[0],
        "channels": X.shape[1],
        "samples": X.shape[2],
        "left": np.sum(y == 0),
        "right": np.sum(y == 1)
    })

validation_df = pd.DataFrame(validation_rows)

validation_df

,subject,trials,channels,samples,left,right
0,1,45,64,481,23,22
1,2,45,64,481,23,22
2,3,45,64,481,23,22
3,4,45,64,481,23,22
4,5,45,64,481,21,24


## Summary

The reusable preprocessing pipeline was successfully tested on Subjects 1–5.

All five subjects produced 45 motor-imagery trials with 64 EEG channels and 481 samples in the +1 to +4 s machine-learning window. Subjects 1–4 contained 23 left and 22 right trials, while Subject 5 contained 21 left and 24 right trials.

The preprocessing pipeline therefore produces consistent data dimensions across the tested subjects. The different class distribution in Subject 5 should be verified before expanding the analysis to the full dataset.

Inspection of Subject 5's original EDF annotations confirmed that each of runs 4, 8, and 12 contains 7 T1 and 8 T2 events. The 21/24 class distribution therefore reflects the original recording rather than an error introduced by preprocessing.

The next stage is to build the first classical machine-learning baseline.